# Auto-supervisione: il segnale è già nei dati

Il codice del capitolo [«Auto-supervisione: il segnale è già nei dati»](https://book.paithon.it/main/AutoSupervisione/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q torch torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

## Auto-supervisione: il segnale è già nei dati

[Leggi la pagina](https://book.paithon.it/main/AutoSupervisione/overview.html)


### Quanta informazione porta una risposta


In [ ]:
from math import log2

# Quanta informazione porta AL PIU' il bersaglio, cioe' la risposta giusta su
# cui il modello si corregge. E' il tetto del canale: quanto ci passa davvero
# e' un'altra domanda.

def bit_per_scelta(n):
    """Una scelta fra n possibilita' equiprobabili vale log2(n) bit."""
    return log2(n)

CLASSI = 1000         # ImageNet: una foto, una categoria fra mille
VOCABOLARIO = 128000  # un tokenizzatore di oggi
FINESTRA = 8192       # token in un esempio di pre-addestramento
PASSI = 10000         # passi di una partita prima del verdetto

etichetta = bit_per_scelta(CLASSI)
token     = bit_per_scelta(VOCABOLARIO)
testo     = token * FINESTRA
rinforzo  = 1.0       # vinto o perso: una risposta binaria per partita

print(f"{'compito':34s} {'bit per esempio':>16s}")
print("-" * 51)
print(f"{'etichetta su ' + str(CLASSI) + ' classi':34s} {etichetta:16.1f}")
print(f"{'un token su ' + str(VOCABOLARIO):34s} {token:16.1f}")
print(f"{'una finestra di ' + str(FINESTRA) + ' token':34s} {testo:16.1f}")
print(f"{'vinto o perso, a fine partita':34s} {rinforzo:16.1f}")
print()
print(f"la finestra di testo vale {testo/etichetta:,.0f} etichette".replace(",", "."))
print(f"e {testo/rinforzo:,.0f} verdetti di fine partita".replace(",", "."))
print(f"spalmato sui {PASSI} passi della partita, il verdetto")
print(f"vale {rinforzo/PASSI:.4f} bit per passo")

## Quattro modi di fabbricare il segnale

[Leggi la pagina](https://book.paithon.it/main/AutoSupervisione/famiglie.html)


### La terza: vincolare le statistiche


In [ ]:
import torch

torch.manual_seed(0)
N, D_IN, D = 512, 32, 8      # esempi, dimensioni in ingresso, coordinate finali

# Due VISTE dello stesso esempio: stesso contenuto, disturbi indipendenti.
contenuto = torch.randn(N, D_IN)
RUMORE_VISTA = 0.3
vista_a = contenuto + RUMORE_VISTA * torch.randn(N, D_IN)
vista_b = contenuto + RUMORE_VISTA * torch.randn(N, D_IN)

# Partenza RIDONDANTE, ed e' il caso interessante: le otto coordinate nascono
# quasi uguali fra loro, cioe' il modello dice otto volte la stessa cosa.
proiettore = torch.nn.Linear(D_IN, D, bias=False)
with torch.no_grad():
    proiettore.weight.copy_(proiettore.weight[0] + 0.05 * torch.randn(D, D_IN))

def correlazione(za, zb):
    """Cross-correlazione fra le due viste, ogni coordinata standardizzata."""
    za = (za - za.mean(0)) / (za.std(0) + 1e-9)
    zb = (zb - zb.mean(0)) / (zb.std(0) + 1e-9)
    return (za.T @ zb) / za.shape[0]

def barlow(c, lam=0.05):
    """Diagonale verso 1 (invarianza), fuori diagonale verso 0 (ridondanza)."""
    diag = torch.diagonal(c)
    fuori = c - torch.diag_embed(diag)
    return ((diag - 1) ** 2).sum() + lam * (fuori ** 2).sum()

def referto(eti, va=None, vb=None):
    va, vb = (vista_a if va is None else va), (vista_b if vb is None else vb)
    with torch.no_grad():
        c = correlazione(proiettore(va), proiettore(vb))
        d, f = torch.diagonal(c), c - torch.diag_embed(torch.diagonal(c))
        print(f"{eti:14s} diagonale {d.mean():5.2f}   "
              f"fuori diagonale {f.abs().sum() / (D * D - D):5.2f}")

# La diagonale non potra' arrivare a 1: le due viste hanno rumore indipendente,
# quindi la loro correlazione ha un tetto, ed e' questo.
print(f"tetto della diagonale, imposto dal rumore: "
      f"{1 / (1 + RUMORE_VISTA ** 2):.2f}\n")

referto("all'inizio")
ott = torch.optim.SGD(proiettore.parameters(), lr=0.05)
for passo in range(1, 601):
    ott.zero_grad()
    barlow(correlazione(proiettore(vista_a), proiettore(vista_b))).backward()
    ott.step()
    if passo in (100, 600):
        referto(f"dopo {passo}")

# Lo stesso proiettore su esempi MAI VISTI. Serve a separare due cose che sulla
# diagonale si confondono: il tetto imposto dal rumore delle viste, e il fatto
# che un proiettore lineare si adatti anche alle 512 righe che ha davanti.
nuovo = torch.randn(4 * N, D_IN)
referto("su dati nuovi",
        nuovo + RUMORE_VISTA * torch.randn(4 * N, D_IN),
        nuovo + RUMORE_VISTA * torch.randn(4 * N, D_IN))

## Perché non collassa, e come si fa a saperlo

[Leggi la pagina](https://book.paithon.it/main/AutoSupervisione/collasso-e-misura.html)


### Che cosa garantisce davvero il punteggio dei metodi contrastivi


In [ ]:
import math
import torch

torch.manual_seed(0)

# Un caso in cui le due viste si corrispondono PERFETTAMENTE: l'informazione
# che l'una porta sull'altra e' tutta quella che c'e', e non e' poca.
# Domanda: quanta ne puo' certificare la InfoNCE, al meglio delle sue
# possibilita'?

def infonce_al_meglio(n, d=64):
    """Loss InfoNCE con un critico PERFETTO fra n candidati, e il limite che segue.

    n e' il numero di candidati fra cui il gioco chiede di scegliere, non la
    dimensione del batch: in NT-Xent un batch di B immagini ne mette 2B-1.
    """
    z = torch.nn.functional.normalize(torch.randn(n, d), dim=1)
    sim = (z @ z.t()) / 0.01           # temperatura bassissima: critico ideale
    perdita = torch.nn.functional.cross_entropy(sim, torch.arange(n)).item()
    # I(x; y) >= log N - L   (van den Oord e colleghi), qui in bit
    return perdita, (math.log(n) - perdita) / math.log(2)

print(f"{'candidati':>18s} {'perdita':>10s} {'bit certificati':>18s} {'log2(N)':>10s}")
for n in (8, 64, 512, 4096):
    perdita, bit = infonce_al_meglio(n)
    print(f"{n:>18d} {perdita:>10.4f} {bit:>18.2f} {math.log2(n):>10.2f}")

## Capire è accorciare

[Leggi la pagina](https://book.paithon.it/main/AutoSupervisione/capire-e-accorciare.html)


### Un fondo esiste, e non tutti lo toccano


In [ ]:
import lzma
import zlib
from collections import defaultdict
from math import log2
from random import Random

# Una lingua con una regola sola: dopo una consonante arriva quasi sempre una
# vocale, e viceversa. La tabella ha somma 1 anche per COLONNE, quindi le
# quattro lettere escono ugualmente frequenti: contarle non serve a niente, e
# tutto quello che c'e' da capire sta nel passaggio da una all'altra.
REGOLA = {
    "a": {"a": 0.05, "e": 0.05, "r": 0.35, "t": 0.55},
    "e": {"a": 0.05, "e": 0.05, "r": 0.55, "t": 0.35},
    "r": {"a": 0.35, "e": 0.55, "r": 0.05, "t": 0.05},
    "t": {"a": 0.55, "e": 0.35, "r": 0.05, "t": 0.05},
}
LETTERE = sorted(REGOLA)
N = 200_000


def genera(n, seme=0):
    """Estrae n lettere dalla sorgente. Il seme e' fissato: il testo non cambia."""
    r, seq = Random(seme), ["a"]
    for _ in range(n - 1):
        p, soglia, cumulata = REGOLA[seq[-1]], r.random(), 0.0
        for lettera in LETTERE:
            cumulata += p[lettera]
            if soglia < cumulata:
                seq.append(lettera)
                break
    return "".join(seq)


def entropia(p):
    return -sum(q * log2(q) for q in p if q > 0)


def bit_per_lettera(testo, ordine):
    """Quanto costa scrivere il testo con un modello che impara leggendolo.

    Non c'e' nessun modello da spedire a parte: chi legge rifa' gli stessi
    conteggi sulle lettere gia' viste, quindi il prezzo di imparare sta DENTRO
    questo numero e non accanto.
    """
    conte = defaultdict(lambda: dict.fromkeys(LETTERE, 1))   # Laplace
    totale = defaultdict(lambda: len(LETTERE))
    bit = 0.0
    for i, lettera in enumerate(testo):
        contesto = testo[max(0, i - ordine):i]
        bit -= log2(conte[contesto][lettera] / totale[contesto])
        conte[contesto][lettera] += 1
        totale[contesto] += 1
    return bit / len(testo)


testo = genera(N)
grezzo = testo.encode("ascii")

# Il fondo: la sorpresa media di una lettera SAPENDO la precedente. Le quattro
# lettere sono equiprobabili, quindi la media sulle righe pesa 1/4 ciascuna.
fondo = sum(entropia(list(REGOLA[s].values())) for s in LETTERE) / len(LETTERE)

# L'oracolo: conosce la tabella dall'inizio e non impara niente. Serve a
# separare due cose che altrimenti si confondono, cioe' quanto costa IMPARARE
# la regola e quanto costa il fatto che proprio QUESTO testo, sorteggiato,
# sia un po' piu' sorprendente della media.
bit = -log2(1 / len(LETTERE))
for prima, dopo in zip(testo, testo[1:]):
    bit -= log2(REGOLA[prima][dopo])
oracolo = bit / len(testo)

print(f"il fondo della sorgente     {fondo:.4f} bit per lettera")
print(f"chi la regola la sapeva     {oracolo:.4f}")
print(f"nessun modello              {log2(len(LETTERE)):.4f}")
for ordine in (0, 1, 2):
    print(f"modello di ordine {ordine}         {bit_per_lettera(testo, ordine):.4f}")
print(f"zlib                        {len(zlib.compress(grezzo, 9)) * 8 / N:.4f}")
print(f"lzma                        {len(lzma.compress(grezzo)) * 8 / N:.4f}")